In [2]:
%pip -q install -U langchain langchain-groq langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [1]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("Groq API key is configured.")

Enter your Groq API key: ··········
Groq API key is configured.


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

response = llm.invoke("Explain what a Large Language Model is in four simple sentences.")

print(response.content)

A Large Language Model is a type of artificial intelligence trained on massive amounts of text data. It learns patterns in language to understand context and generate human-like responses. By predicting the next likely word in a sequence, it can write, summarize, and answer questions. This technology powers many modern chatbots and writing assistants.


In [6]:
print("Response type:", type(response))
print("Generated text:", response.content)
print("Response metadata:", response.response_metadata)

Response type: <class 'langchain_core.messages.ai.AIMessage'>
Generated text: A Large Language Model is a type of artificial intelligence trained on massive amounts of text data. It learns patterns in language to understand context and generate human-like responses. By predicting the next likely word in a sequence, it can write, summarize, and answer questions. This technology powers many modern chatbots and writing assistants.
Response metadata: {'token_usage': {'completion_tokens': 64, 'prompt_tokens': 25, 'total_tokens': 89, 'completion_time': 0.234363192, 'completion_tokens_details': None, 'prompt_time': 0.001537486, 'prompt_tokens_details': None, 'queue_time': 0.076954415, 'total_time': 0.235900678}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_2f6fa82922', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}


In [8]:
# In Google Colab, upload a PDF when prompted.
# If you are running locally, set PDF_PATH to your file path instead.

PDF_PATH = None

try:
    from google.colab import files
    uploaded = files.upload()
    pdf_files = [name for name in uploaded if name.lower().endswith(".pdf")]
    if pdf_files:
        PDF_PATH = pdf_files[0]
except ImportError:
    print("Not running in Google Colab. You can set PDF_PATH manually.")

In [24]:
from langchain_core.documents import Document

if PDF_PATH:
    from langchain_community.document_loaders import PyPDFLoader

    loader = PyPDFLoader(PDF_PATH)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from {PDF_PATH}")
else:
    # Small example knowledge base used when no PDF is uploaded.
    sample_policy = """
    UNIVERSITY PROJECT SUBMISSION POLICY

    Standard deadline:
    Students must submit their project before 11:59 PM on the announced deadline.

    Late submission:
    A submission up to two calendar days late requires prior approval from the course coordinator.
    An approved submission within this two-day period receives a 10 percent penalty.

    More than two days late:
    Submissions more than two calendar days late are normally not accepted.
    Exceptions require written approval from the department head.

    Extension requests:
    Students should email the course coordinator before the deadline.
    The request should include the course name, project title, reason, and requested extension date.

    Attendance:
    Students must maintain at least 75 percent attendance to be eligible for the final assessment.
    Medical exceptions are reviewed by the department with supporting documents.

    Scolding Co-ordinator/lecture:if the student try to scold the lecture or any faculty for they asking student to submit the report
    they will be disqualified from the final assessment.
    they will not be able to attend the final examination.
    they should bring their parent and need to pay penalty upto 50k and they will be suspened from the college
    """

    documents = [
        Document(
            page_content=sample_policy,
            metadata={"source": "sample_university_policy", "page": 1}
        )
    ]

print("Number of documents:", len(documents))
print("\nFirst document preview:\n")
print(documents[0].page_content[:1000])
print("\nMetadata:", documents[0].metadata)

Number of documents: 1

First document preview:


    UNIVERSITY PROJECT SUBMISSION POLICY

    Standard deadline:
    Students must submit their project before 11:59 PM on the announced deadline.

    Late submission:
    A submission up to two calendar days late requires prior approval from the course coordinator.
    An approved submission within this two-day period receives a 10 percent penalty.

    More than two days late:
    Submissions more than two calendar days late are normally not accepted.
    Exceptions require written approval from the department head.

    Extension requests:
    Students should email the course coordinator before the deadline.
    The request should include the course name, project title, reason, and requested extension date.

    Attendance:
    Students must maintain at least 75 percent attendance to be eligible for the final assessment.
    Medical exceptions are reviewed by the department with supporting documents.
    
    Scolding Co-ordinator/l

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = text_splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Number of chunks:", len(chunks))
print("\nExample chunk:\n")
print(chunks[0].page_content)
print("\nChunk metadata:", chunks[0].metadata)

Original documents: 1
Number of chunks: 2

Example chunk:

UNIVERSITY PROJECT SUBMISSION POLICY

    Standard deadline:
    Students must submit their project before 11:59 PM on the announced deadline.

    Late submission:
    A submission up to two calendar days late requires prior approval from the course coordinator.
    An approved submission within this two-day period receives a 10 percent penalty.

    More than two days late:
    Submissions more than two calendar days late are normally not accepted.
    Exceptions require written approval from the department head.

    Extension requests:
    Students should email the course coordinator before the deadline.
    The request should include the course name, project title, reason, and requested extension date.

Chunk metadata: {'source': 'sample_university_policy', 'page': 1}


In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Convert one text chunk into a vector to inspect the result.
example_vector = embedding_model.embed_query(chunks[0].page_content)

print("Vector type:", type(example_vector))
print("Vector length:", len(example_vector))
print("First 10 values:", example_vector[:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector type: <class 'list'>
Vector length: 384
First 10 values: [0.006444619502872229, -0.007115598302334547, 0.0269861351698637, 0.0036127655766904354, 0.04132834076881409, -0.03255452215671539, -0.06842377036809921, -0.07096961885690689, 0.005190275143831968, 0.09036076813936234]


In [27]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS vector store created.")
print("Indexed chunks:", len(chunks))

FAISS vector store created.
Indexed chunks: 2


In [28]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 1}
)

question = "what will happen if i scold the lecture for they asking me to submit the report?"

retrieved_docs = retriever.invoke(question)

print("Retrieved passages:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\\n--- Retrieved chunk {i} ---")
    print("Source metadata:", doc.metadata)
    print(doc.page_content)

Retrieved passages: 1
\n--- Retrieved chunk 1 ---
Source metadata: {'source': 'sample_university_policy', 'page': 1}
Attendance:
    Students must maintain at least 75 percent attendance to be eligible for the final assessment.
    Medical exceptions are reviewed by the department with supporting documents.
    
    Scolding Co-ordinator/lecture:if the student try to scold the lecture or any faculty for they asking student to submit the report
    they will be disqualified from the final assessment.
    they will not be able to attend the final examination.
    they should bring their parent and need to pay penalty upto 50k and they will be suspened from the college


In [29]:
# Join the retrieved passages into one context string.
context = "\\n\\n".join(doc.page_content for doc in retrieved_docs)

from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful university policy assistant.

Answer the question using only the context below.
If the context does not contain the answer, clearly say that
the policy information could not be found. Do not invent rules.

Context:
{context}

Question:
{question}

Answer:
"""
)

prompt_value = rag_prompt.invoke({
    "context": context,
    "question": question
})

print(prompt_value.to_string())
rag_response = llm.invoke(prompt_value)

print(rag_response.content)

Human: 
You are a helpful university policy assistant.

Answer the question using only the context below.
If the context does not contain the answer, clearly say that
the policy information could not be found. Do not invent rules.

Context:
Attendance:
    Students must maintain at least 75 percent attendance to be eligible for the final assessment.
    Medical exceptions are reviewed by the department with supporting documents.
    
    Scolding Co-ordinator/lecture:if the student try to scold the lecture or any faculty for they asking student to submit the report
    they will be disqualified from the final assessment.
    they will not be able to attend the final examination.
    they should bring their parent and need to pay penalty upto 50k and they will be suspened from the college

Question:
what will happen if i scold the lecture for they asking me to submit the report?

Answer:

Based on the provided policy, if you scold a lecture for asking you to submit a report, the followi

In [30]:
rag_response = llm.invoke(prompt_value)

print(rag_response.content)

Based on the provided policy, if you scold a lecture for asking you to submit a report, the following consequences will apply:

*   You will be **disqualified from the final assessment**.
*   You will **not be able to attend the final examination**.
*   You must **bring your parent**.
*   You are required to **pay a penalty of up to 50k**.
*   You will be **suspended from the college**.


In [33]:
def ask_rag(question):
    # Retrieve relevant document chunks
    retrieved_docs = retriever.invoke(question)

    # Join the retrieved passages into one context string
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # Invoke the RAG prompt template
    prompt_value = rag_prompt.invoke({
        "context": context,
        "question": question
    })

    # Get the LLM response
    response = llm.invoke(prompt_value)
    return response.content, retrieved_docs

test_questions = [
    "What is the penalty for an approved late submission?",
    "What attendance is required for the final assessment?",
    "Can a student submit a project five days late?",
    "What is the university's hostel fee?",
    "what will happen if the student scold the faculty?"
]

for q in test_questions:
    answer, _ = ask_rag(q)
    print("\nQUESTION:", q)
    print("ANSWER:", answer)
    print("-" * 80)


QUESTION: What is the penalty for an approved late submission?
ANSWER: The penalty for an approved late submission is 10 percent.
--------------------------------------------------------------------------------

QUESTION: What attendance is required for the final assessment?
ANSWER: Students must maintain at least 75 percent attendance to be eligible for the final assessment.
--------------------------------------------------------------------------------

QUESTION: Can a student submit a project five days late?
ANSWER: Based on the policy, a student **cannot** normally submit a project five days late, as submissions more than two calendar days late are normally not accepted. However, an exception is possible if the student obtains **written approval from the department head**.
--------------------------------------------------------------------------------

QUESTION: What is the university's hostel fee?
ANSWER: The policy information could not be found.
------------------------------

In [35]:
import json

# 1. Define a simple mathematical tool
def add_numbers(a: int, b: int) -> int:
    print(f"\n[Tool Executing] Adding {a} + {b}...")
    return a + b

# 2. Build a clear, student-friendly agent loop
def simple_agent(user_question: str):
    print(f"User Question: '{user_question}'")

    # Instruct the model to reason whether it needs to call a tool
    prompt = f"""
    You are an AI Agent with access to one tool: 'add_numbers(a, b)'.

    If the question requires adding numbers, output a JSON block like this:
    {{"action": "add_numbers", "args": [number1, number2]}}

    Otherwise, answer the question directly.

    Question: {user_question}
    Answer/Action:
    """

    # Step 1: Reason / Decide
    print("\n[Agent Reasoning] Thinking...")
    llm_decision = llm.invoke(prompt).content.strip()
    print(f"[Agent Decision] {llm_decision}")

    # Step 2: Act
    if "add_numbers" in llm_decision:
        try:
            # Simple parsing of the tool arguments from the LLM output
            start_idx = llm_decision.find("{")
            end_idx = llm_decision.rfind("}") + 1
            action_json = json.loads(llm_decision[start_idx:end_idx])

            a, b = action_json["args"]
            # Execute the tool
            tool_result = add_numbers(a, b)

            # Step 3: Formulate final response with the tool's result
            print("\n[Agent Reasoning] Formulating final answer based on tool output...")
            final_prompt = f"""
            The user asked: {user_question}
            You executed the tool 'add_numbers' and got the result: {tool_result}
            Now, output the final friendly answer directly.
            """
            final_answer = llm.invoke(final_prompt).content.strip()
            return final_answer
        except Exception as e:
            return f"Could not process tool execution. Error: {e}"
    else:
        return llm_decision

# Test the agent with a math query!
result = simple_agent("How much is 142512345 plus 3125654859?")
print("\nFinal Agent Answer:\n", result)

User Question: 'How much is 142512345 plus 3125654859?'

[Agent Reasoning] Thinking...
[Agent Decision] {"action": "add_numbers", "args": [142512345, 3125654859]}

[Tool Executing] Adding 142512345 + 3125654859...

[Agent Reasoning] Formulating final answer based on tool output...

Final Agent Answer:
 142,512,345 plus 3,125,654,859 equals **3,268,167,204**.
